# Approximations

In [ ]:
import numpy as np

import plotly.graph_objects as go

from scipy.stats import norm, uniform, f, gaussian_kde
from sklearn.neighbors import KernelDensity
from scipy import interpolate

In [ ]:
from yahooquery import Ticker

Take true dependency $y=2x-x^2$ and generate points with noise $y_i = 2x_i-x_i^2+\varepsilon_i,\,\varepsilon_i\sim N(0,\sigma^2)$.<br> 
Can Kernel regression reconstruct true dependency?

In [ ]:
sigma = 0.4
n_points = 200
x_given = uniform.rvs(scale = 1.5, size = n_points)
epsilon = norm.rvs(scale = sigma, size = n_points)
y_given = 2*x_given - x_given**2 + epsilon

In [ ]:
plot = go.Figure()

x_ = np.linspace(0,1.5,1000)
graph1 = go.Scatter(x = x_, y = 2*x_-x_**2)
graph2 = go.Scatter(x = x_given, y = y_given, mode = 'markers')
plot.add_trace(graph1)
plot.add_trace(graph2)

plot.show()

In [ ]:
def gaus_kernel(x,xi,h):
    return np.exp(-0.5*(x-xi)**2/h**2)

def kd_regression(x, x_data, y_data, h):
    k_h = gaus_kernel(x, x_data, h)
    
    numerator = np.dot(y_data, k_h)
    denominator = np.sum(k_h)
    return numerator/denominator

In [ ]:
# gaus_kernel(x_[0], x_given, 1)
# kd_regression(x_[0], x_given, y_given, 1)

In [ ]:
h=0.08
x_ = np.linspace(0,1.5,1000)

y_estimated = [kd_regression(xi,x_given,y_given,h) for xi in x_]

plot = go.Figure()

graph1 = go.Scatter(x = x_, y = 2*x_-x_**2)
graph2 = go.Scatter(x = x_given, y = y_given, mode = 'markers')
graph3 = go.Scatter(x = x_, y = y_estimated, name = 'Kernel')
plot.add_trace(graph1)
plot.add_trace(graph3)
plot.add_trace(graph2)

plot.show()

<H4>Kernel density estimation (KDE)</H4>
https://en.wikipedia.org/wiki/Kernel_density_estimation

scikit-learn<br>
https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KernelDensity.html#sklearn.neighbors.KernelDensity<br>
https://scikit-learn.org/stable/auto_examples/neighbors/plot_kde_1d.html

<u>Unimodal distribution.</u> Take a sample of Fisher $F(5,10)$ distribution and approximate pdf using KDE.  

In [ ]:
df_1 = 5
df_2 = 10 
sample = f.rvs(df_1,df_2, size = 1000)

1. scipy.stats.gaussian_kde<br>
https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.gaussian_kde.html

In [ ]:
# kernel_pdf = gaussian_kde(sample, bw_method = 0.008)
kernel_pdf = gaussian_kde(sample)

x_ = np.linspace(-0.1,10,1000)
y_ = kernel_pdf.evaluate(x_)

plot = go.Figure()
graph = go.Scatter(x = x_, y = f.pdf(x_,df_1,df_2))
graph2 = go.Scatter(x = x_, y = y_, name = 'KDE')
hist = go.Histogram(x=sample, histnorm = 'probability density')
plot.add_trace(hist)
plot.add_traces([graph, graph2])

2. scikit-learn<br>
https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KernelDensity.html#sklearn.neighbors.KernelDensity<br>
https://scikit-learn.org/stable/auto_examples/neighbors/plot_kde_1d.html

In [ ]:
sample_modif = np.array(sample).reshape(-1, 1)
kernel = KernelDensity(kernel="gaussian", bandwidth=0.8).fit(sample_modif)
kernel.get_params()

In [ ]:
x_modif = np.array(x_).reshape(-1,1)
# x_modif

In [ ]:
log_y_reg = kernel.score_samples(x_modif)
y_reg = [np.exp(log_yi) for log_yi in log_y_reg]

In [ ]:
x_ = np.linspace(-0.1,10,1000)

plot = go.Figure()
graph = go.Scatter(x = x_, y = f.pdf(x_,df_1,df_2))
hist = go.Histogram(x=sample, histnorm = 'probability density')
graph2 = go.Scatter(x = x_, y = y_reg, name = 'KDE')
plot.add_trace(hist)
plot.add_trace(graph)
plot.add_trace(graph2)

<H3>Interpolation and Splines</H3> 

Take Treasury yields
https://www.bloomberg.com/markets/rates-bonds/government-bonds/us

In [ ]:
time_ = [0.25, 0.5, 1, 2, 5, 10, 30]
yield_ = [3.69, 3.7, 3.67, 3.75, 3.88, 4.27, 4.89]

plot = go.Figure()
graph = go.Scatter(x = time_, y = yield_, mode = 'markers')
plot.add_trace(graph)

These values are exact ones. So fitting curve should go exactly through given points.<br>
https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.interp1d.html#scipy.interpolate.interp1d

In [ ]:
res_func = interpolate.interp1d(x = time_, y = yield_, kind = 'quadratic')
# try kind = 'quadratic', 'cubic'

In [ ]:
t_ = np.linspace(0.25,30,1001)
y_ = res_func(t_)

plot = go.Figure()
graph = go.Scatter(x = time_, y = yield_, mode = 'markers')
graph2 = go.Scatter(x = t_, y = y_, name ='Interp')
plot.add_traces([graph, graph2])

**KDE on Real data**. Choose time period to analyse trajectory of Coca-cola prices.

In [ ]:
def kernel_pdf(sample_, x_, h):
    sample_modif = np.array(sample_).reshape((-1, 1))
    kernel = KernelDensity(kernel="gaussian", bandwidth=h).fit(sample_modif)

    x_modif = np.array(x_).reshape((-1,1))
    log_y_reg = kernel.score_samples(x_modif)
    y_reg = [np.exp(log_yi) for log_yi in log_y_reg]
    
    return y_reg

In [ ]:
cola = Ticker('KO')
cola_data = cola.history(start='2023-01-01', end = '2026-04-16', interval = '1d')

In [ ]:
t = cola_data.index.get_level_values(level=1)
prices = cola_data['close']

plot = go.Figure()
graph = go.Scatter(x = t, y = prices)
plot.add_trace(graph)

Check histogram and fit distribution for prices

In [ ]:
x_ = np.linspace(50, 75,500)
y_pdf = kernel_pdf(prices, x_,0.5)

plot = go.Figure()
hist = go.Histogram(x = prices, histnorm = 'probability density')
plot.add_trace(hist)
graph = go.Scatter(x = x_, y = y_pdf)
plot.add_trace(graph)